In [ ]:
# Julia script for extracting the  ring parameters (ring_size, ring_width, position angle) from the image blurred by Picasso

In [1]:
using CSV
using DataFrames
using DelimitedFiles
using Picasso

In [2]:
using VLBIImagingSummaryStats
using VLBISkyModels
using ComradeBase
using ImageFiltering

In [3]:
 function intensityfield_to_img(intensityfield, freq, config)
            img = zeros(Int.(sqrt(config["tot_blocks"] * config["tot_pixels"])), Int.(sqrt(config["tot_blocks"] * config["tot_pixels"])))
            for f in freq
                if f == config["IMG_FREQ"]
                    for block in 1:config["tot_blocks"]
                        for pixel in 1:config["tot_pixels"]
                            jj = Int.(div(block - 1, sqrt(config["tot_blocks"])) * sqrt(config["tot_pixels"]) + div(pixel - 1, sqrt(config["tot_pixels"])) + 1)
                            ii = Int.((rem(block - 1, sqrt(config["tot_blocks"]))) * sqrt(config["tot_pixels"]) + rem(pixel - 1, sqrt(config["tot_pixels"])) + 1)
                            img[ii, jj] = intensityfield[block].IQUV[pixel, 1, 1] * intensityfield[block].dx[1] * intensityfield[block].dx[2] * Picasso.JANSKY_FACTOR
                        end
                    end
                    break
                end
            end
            return img

 end

intensityfield_to_img (generic function with 1 method)

In [4]:
 function psize_rad(config)
        nx = config["IMG_WIDTH"]
        ny = config["IMG_HEIGHT"]

        src_dist_rg = config["SOURCE_DIST"] * Picasso.SPEED_OF_LIGHT^2 / (Picasso.GGRAV * config["MBH"])

        # Angular size of each pixel (rad/pixel)
        psizex = config["CAM_SIZE_IMG"] / (nx * src_dist_rg)
        psizey = config["CAM_SIZE_IMG"] / (ny * src_dist_rg)

        return src_dist_rg, psizex, psizey

 end 

psize_rad (generic function with 1 method)

In [5]:
function blur_image(config, img)
        src_dist_rg, psizex, psizey = psize_rad(config)
        psize_muas = rad_to_muas(psizex)
        blur_FWHM = 20.0
        blur_muas = blur_FWHM / sqrt(8.0 * log(2.0))
        sigma_pixels = blur_muas / psize_muas
        img_blur = imfilter(img, Kernel.gaussian(sigma_pixels))
        return img_blur
end

blur_image (generic function with 1 method)

In [6]:
function img_VIDA(img, config)
        img_blur = blur_image(config, img)

        nx = config["IMG_WIDTH"]
        ny = config["IMG_HEIGHT"]
        # Angular size of each pixel (rad/pixel)
        src_dist_rg, psizex, psizey = psize_rad(config)

        # Center pixel of the image
        x0c = 0.0
        y0c = 0.0

        # Observation time
        mjd = config["obs_time"]

        # RA, DEC and position angle (M87 values)
        ra = config["SOURCE_RA"]
        dec = config["SOURCE_DEC"]
        # pa = config["SOURCE_pa"] * pi / 180.0
        pa = 0.0

        # Frequency
        freq = config["IMG_FREQ"]

        # Source name
        source = "M87"

        # Put into Comrade format
        info = ComradeBase.MinimalHeader(source, ra, dec, mjd, freq)
        g = imagepixels(psizex * nx, psizey * ny, nx, ny, x0c * psizex, y0c * psizey;
        header=info, posang=pa)
        imap = IntensityMap(img_blur[end:-1:begin, :], g)
   
        return g, imap
end

img_VIDA (generic function with 1 method)

In [7]:
rad_to_muas(rad) = rad * 180.0 * 3.6e9 / pi
muas_to_rad(muas) = muas * pi / (180.0 * 3.6e9)

muas_to_rad (generic function with 1 method)

In [8]:
function fit_image_SumStat(intensityfield, freq, config, fit_results)
    #  intensityfield---->ehtim
    img = intensityfield_to_img(intensityfield, freq, config)
    grid, imap = img_VIDA(img, config)
    cent_img = center_template(imap, MRing{3})

    M87_ring_2017 = muas_to_rad(42) / 2.0
    M87_ring_sig = muas_to_rad(3) / 2.0
    M87_PA = 175.0
    M87_PA_sig = 25.0
    M87_width = muas_to_rad(20.0)
    M87_width_sig = muas_to_rad(10.0)

    ring_size = cent_img[2].r0
    ring_width = cent_img[2].σ
    PA = cent_img[2].ξ[1] * 180.0 / pi

    @info "Ring size = $(round(rad_to_muas(ring_size), digits = 5)), Ring width = $(round(rad_to_muas(ring_width), digits = 5)), PA = $(round((cent_img[2].ξ[1]) * 180.0 / pi, digits = 5))"

    width_chi2 = 0.0
    img_chi2 = 0.0

    ring_chi2 = ((ring_size - M87_ring_2017) / M87_ring_sig)^2
    PA_chi2 = ((PA - M87_PA) / M87_PA_sig)^2
    width_chi2 = ring_width <= 20 ? 0 : ((ring_width - M87_width) / M87_width_sig)^2

    img_chi2 = ring_chi2 + PA_chi2 + width_chi2

    fit_results.ring_size[1] = rad_to_muas(ring_size)
    fit_results.ring_width[1] = rad_to_muas(ring_width)
    fit_results.PA[1] = PA
    fit_results.img_chi2[1] = img_chi2

    return img_chi2, ring_size, ring_width, PA

end

fit_image_SumStat (generic function with 1 method)

In [9]:
# function to run multiple GRRT runs and saving the parameters to a csv file
function redo(n0; csvfile="/home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv")

   
    println("Writing to: ", abspath(csvfile))
    input = PicassoSetup()
    input.config["IMG_HEIGHT"] = n0
    input.config["IMG_WIDTH"]  = n0
    intensityfield = PicassoGRRT(input; n0=0) # for fitting the parameters need to set the ray tracing parameters
                                               #  manually in config for each resolution run
    for i in 1:50
        println("Run $i:")
       
        img= intensityfield_to_img(intensityfield,100.e9,input.config)
        src_dist_rg, psizex, psizey= psize_rad(input.config)
        img_blur= blur_image(input.config, img)
        g, imap = img_VIDA(img, input.config)
   
        

        img_chi2, ring_size, ring_width, PA =fit_image_SumStat(intensityfield, 100.e9, input.config, input.fit_results)
    
        row = DataFrame(
            run        = [i],
            n0         = [n0],
            ring_size  = [round(rad_to_muas(ring_size),digits=5)],
            ring_width = [round(rad_to_muas(ring_width),digits =5)],
            pa         = [round(PA, digits=5)]
        )

        # Write immediately (important for long runs / crashes)
        if isfile(csvfile)
            CSV.write(csvfile, row; append=true)
        else
            CSV.write(csvfile, row)
        end

     println("Finished run $i\n")   
    end
    
    return nothing
    
end

redo (generic function with 1 method)

In [10]:
redo(50)

Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Ring size = 18.99842, Ring width = 11.86187, PA = 147.12868
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/fit_image.jl:398
┌ Info: Image chi^2 = 3.02349, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158
┌ Info: Ring size = 19.01369, Ring width = 11.84876, PA = 146.83841
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Run 2:
Run 3:


...

Run 49:


┌ Info: Ring size = 19.01043, Ring width = 11.89504, PA = 147.29027
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Run 50:


┌ Info: Ring size = 19.01613, Ring width = 11.85504, PA = 147.10076
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18
┌ Info: Ring size = 18.98169, Ring width = 11.8693, PA = 147.14878
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


UndefVarError: UndefVarError: `i` not defined

In [13]:
redo(100)

Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Ring size = 19.49264, Ring width = 11.64082, PA = 149.00953
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/fit_image.jl:398
┌ Info: Image chi^2 = 2.09064, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158


Finished run 1

Run 2:


...

┌ Info: Ring size = 19.6114, Ring width = 11.63521, PA = 150.49118
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.50002, Ring width = 11.63926, PA = 148.8338
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.52224, Ring width = 11.62757, PA = 149.37296
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


In [14]:
redo(150)


Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Ring size = 19.55069, Ring width = 11.42432, PA = 147.80137
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/fit_image.jl:398
┌ Info: Image chi^2 = 2.11718, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158


Finished run 1

Run 2:


...

┌ Info: Ring size = 19.51158, Ring width = 11.42406, PA = 147.77253
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.54068, Ring width = 11.43343, PA = 147.90041
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.51202, Ring width = 11.38906, PA = 148.4822
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


In [15]:
redo(200)


Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Image chi^2 = 1.97214, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158


Finished run 1

Run 2:


...

┌ Info: Ring size = 19.76942, Ring width = 11.32742, PA = 146.40194
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.72016, Ring width = 11.32954, PA = 146.23718
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.74577, Ring width = 11.30869, PA = 146.23292
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


In [16]:
redo(250)


Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Image chi^2 = 1.9353, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158


Finished run 1

Run 2:


...

┌ Info: Ring size = 19.70213, Ring width = 11.1839, PA = 149.26297
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.69892, Ring width = 11.21913, PA = 148.0017
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.71722, Ring width = 11.18343, PA = 147.90398
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


In [17]:
redo(300)

Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Image chi^2 = 1.91642, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158


Finished run 1

Run 2:


...

┌ Info: Ring size = 19.74671, Ring width = 11.36398, PA = 148.55243
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.61452, Ring width = 11.36702, PA = 148.44272
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.64325, Ring width = 11.35935, PA = 148.34157
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


In [10]:
redo(350)

Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Image chi^2 = 1.71764, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158
┌ Info: Ring size = 19.66264, Ring width = 11.32038, PA = 148.05318
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 1

Run 2:
Finished run 2

Run 3:


...

┌ Info: Ring size = 21.21359, Ring width = 11.91636, PA = 147.93978
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.66302, Ring width = 11.3268, PA = 148.04204
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.67336, Ring width = 11.32836, PA = 148.08031
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


In [11]:
redo(400)

Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Image chi^2 = 1.93014, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158


Finished run 1

Run 2:


...

┌ Info: Ring size = 19.69044, Ring width = 11.22548, PA = 147.68848
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.69746, Ring width = 11.25665, PA = 147.93189
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.75414, Ring width = 11.28871, PA = 147.80291
└ @ Main /home/amandeep-kaur/Downloads/trials/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


In [10]:
redo(500)

Writing to: /home/amandeep-kaur/aspire-black-hole-project/Data_files/Blurring_ring_stats.csv


┌ Info: Initializing Picasso...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:11
┌ Info: Setup complete!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:31
┌ Info: Setting M_UNIT...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:247
┌ Info: Finished! Int. flux (230 GHz) = 0.62527
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:339
┌ Info: M_UNIT = 3.19021048687802e25
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/setup.jl:340
┌ Info: Generating image...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:72
┌ Info: Finished GRRT!
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:107
┌ Info: Fitting image and spectrum (M87 2017)...
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:108
┌ Info: Integrated flux density = 0.0 Jy
└ @ Picasso /home/ama

Run 1:


┌ Info: Image chi^2 = 1.77611, Spectrum chi^2 = 0.0
└ @ Picasso /home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/Picasso.jl:158
┌ Info: Ring size = 19.7727, Ring width = 11.11489, PA = 149.04475
└ @ Main /home/amandeep-kaur/aspire-black-hole-project/notebook/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 1

Run 2:
Finished run 2

Run 3:


...

┌ Info: Ring size = 21.55766, Ring width = 11.88953, PA = 147.0922
└ @ Main /home/amandeep-kaur/aspire-black-hole-project/notebook/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 49

Run 50:


┌ Info: Ring size = 19.6803, Ring width = 11.18747, PA = 147.79837
└ @ Main /home/amandeep-kaur/aspire-black-hole-project/notebook/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18


Finished run 50



┌ Info: Ring size = 19.75215, Ring width = 11.17857, PA = 147.81383
└ @ Main /home/amandeep-kaur/aspire-black-hole-project/notebook/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X11sZmlsZQ==.jl:18
